В цьому домашньому завданні кожне завдання оцінюється по 10 балів.

 **Завдання 1.** Після перегляду лекцій про поняття функії, вступ до лінійної алгебри і мат. формулювання лін. регресії знайдіть найкращу лінію для прогнозу `charges` за `age` **для некурців** (датафрейм `non_smoker_df`) з допомогою

1. Методу МНК (з використанням тільки `numpy`, без `scikit learn`)

2. Full-Batch градієнтного спуску з `numpy` . Протестуйте 3 різних learning rate і зробіть висновок, який є найкращим виходячи з практик для цього, наведених в лекції. Зверніть увагу, що на вхід треба набір даних дворозміний, для цього можливо треба буде трансформувати Ваші дані X в формат, як був в лекції "Математичне формулювання лінійної регресії". Також, градієнтний спуск в нашому випадку може розходитись з навчальним рейтом 0.1, бо цей рейт в цій задачі завеликий. Спробуйте нижчі рейти.
3. З `scikit-learn.LinearRegression`. Тут зверніть увагу, що вхід `X` має бути двовимірним масивом, тому нам потрібно передати dataframe, а не окрему колонку. Якщо у Вас X - колонка (а у Вас так мало б бути), то можна скористатись `X.to_frame()` щоб конвертувати колонку в датафрейм.

Для кожного методу
- знайдіть і виведіть коефіцієнти моделі
- обчисліть прогнози моделі і збережіть в окрему змінну
- порахуйте точність прогнозу RMSE  

Для градієнтного спуску виведіть графік помилки в залежності від ітерації.

А також побудуйте на одному графіку дані `age` проти `charges` в вигляді діаграми розсіювання і всі три лінії регресії, знайдені кожним з методів (для град. спуску оберіть варіант з тим learning rate, який виявився найкращим).

Зробіть висновки, чи відрізняються результати моделей?
Чи є знайдены параметри моделы близькими до ваших найкращих припущень?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

from ml_homework.metrics import root_mean_squared_error as rmse
from ml_homework.optimization import full_batch_gradient_descent
from ml_homework.paths import RAW_DATA_DIR
from ml_homework.visualization import plot_regression_predictions

medical_df = pd.read_csv(RAW_DATA_DIR / "medical/medical-charges.csv")
non_smoker_df = medical_df.loc[medical_df["smoker"] == "no"].copy()

In [ ]:
non_smoker_df

In [ ]:
x = non_smoker_df["age"].values
y = non_smoker_df["charges"].values

w_mnk = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
b_mnk = y.mean() - w_mnk * x.mean()

mnk_predictions = w_mnk * x + b_mnk
mnk_rmse = rmse(y, mnk_predictions)

print("w:", w_mnk)
print("b:", b_mnk)
print("RMSE:", mnk_rmse)
print("mnk_predictions:", mnk_predictions)

In [ ]:
plot_regression_predictions(
    x,
    y,
    {"МНК лінія регресії": mnk_predictions},
    title="Лінійна регресія: age vs charges для некурців",
    xlabel="Age",
    ylabel="Charges",
)
plt.show()

In [ ]:
X = non_smoker_df["age"].to_frame().to_numpy()
Y = non_smoker_df["charges"].to_numpy()
X, Y

In [ ]:
learning_rates = [0.000001, 0.00001, 0.0001]

gd_results = []
gd_predictions = {}
gd_errors = {}

for lr in learning_rates:
    w_gd, b_gd, errors_gd = full_batch_gradient_descent(
        X, Y, learning_rate=lr, epochs=10000
    )

    predictions_gd = w_gd * X[:, 0] + b_gd
    rmse_gd = rmse(Y, predictions_gd)

    gd_results.append(
        {
            "learning_rate": lr,
            "w": w_gd,
            "b": b_gd,
            "RMSE": rmse_gd,
        }
    )

    gd_predictions[lr] = predictions_gd
    gd_errors[lr] = errors_gd

In [ ]:
gd_results_df = pd.DataFrame(gd_results)
gd_results_df

In [ ]:
best_result = gd_results_df.loc[gd_results_df["RMSE"].idxmin()]

best_lr = best_result["learning_rate"]
best_w_gd = best_result["w"]
best_b_gd = best_result["b"]
best_rmse_gd = best_result["RMSE"]

best_gd_predictions = gd_predictions[best_lr]

print("Best learning rate:", best_lr)
print("w:", best_w_gd)
print("b:", best_b_gd)
print("RMSE:", best_rmse_gd)

In [ ]:
plt.figure(figsize=(10, 6))

for lr in learning_rates:
    errors = gd_errors[lr]

    plt.plot(range(0, len(errors), 100), errors[::100], label=f"learning rate = {lr}")

plt.xlabel("Iteration")
plt.ylabel("RMSE")
plt.title("Full-Batch Gradient Descent: RMSE залежно від ітерації")
plt.legend()
plt.grid(True)
plt.show()

Було протестовано три значення learning rate: 0.000001, 0.00001 та 0.0001.

Найкращий результат серед протестованих показав learning rate = 0.0001, оскільки після 10 000 ітерацій він дав найменший RMSE — приблизно 4696.33.

Для цього learning rate модель отримала коефіцієнти w ≈ 229.61 та b ≈ -419.83. Ці параметри ще відрізняються від результатів методу найменших квадратів (w ≈ 267.25, b ≈ -2091.42), тому градієнтний спуск ще не повністю зійшовся до оптимального розв’язку.

Менші значення learning rate навчаються повільніше, тому після однакової кількості ітерацій мають більший RMSE.

Отже, серед протестованих значень найкращим є learning rate = 0.0001.

In [ ]:
lin_reg = LinearRegression()

In [ ]:
lin_reg.fit(X, Y)

In [ ]:
lin_reg.coef_, lin_reg.intercept_

In [ ]:
predictions_sklearn = lin_reg.predict(X)

In [ ]:
sklearn_rmse = rmse(Y, predictions_sklearn)

print("w:", lin_reg.coef_[0])
print("b:", lin_reg.intercept_)
print("RMSE:", sklearn_rmse)

In [ ]:
plot_regression_predictions(
    X[:, 0],
    Y,
    {
        "MNK": mnk_predictions,
        "Full-batch Gradient Descent": best_gd_predictions,
        "Sklearn": predictions_sklearn,
    },
    title="Comparison of Linear Regression and Gradient Descent",
    xlabel="Age",
    ylabel="Charges",
)
plt.show()

 **Завдання 2.** Навчіть модель лінійної регресії з допомогою sklearn оцінювати розмір медичних збори для **курців** за їх віком.
Виведіть
- точність моделі
-  коефіцієнти
-  візуалізуйте модель у вигляді лінії на графіку розсіювання `age` проти `charges`

і зробіть висновки, чи це хороша модель, чи ви б її використовували в компанії?

In [ ]:
smoker_df = medical_df[medical_df.smoker == "yes"]

In [ ]:
smoker_df

In [ ]:
X_smoker = smoker_df["age"].to_frame().to_numpy()
Y_smoker = smoker_df["charges"].to_numpy()

In [ ]:
smoker_lin_reg = LinearRegression()

In [ ]:
smoker_lin_reg.fit(X_smoker, Y_smoker)

In [ ]:
smoker_lin_reg.coef_, smoker_lin_reg.intercept_

In [ ]:
smoker_predictions_sklearn = smoker_lin_reg.predict(X_smoker)

In [ ]:
smoker_sklearn_rmse = rmse(Y_smoker, smoker_predictions_sklearn)

print("w:", smoker_lin_reg.coef_[0])
print("b:", smoker_lin_reg.intercept_)
print("RMSE:", smoker_sklearn_rmse)

In [ ]:
plot_regression_predictions(
    X_smoker[:, 0],
    Y_smoker,
    {"Sklearn": smoker_predictions_sklearn},
    title="Linear Regression for Smokers: Age vs Charges",
    xlabel="Age",
    ylabel="Charges",
)
plt.show()

Модель лінійної регресії для курців отримала коефіцієнт w ≈ 305.24 та вільний член b ≈ 20294.13. Це означає, що зі збільшенням віку на один рік прогнозовані медичні витрати курця в середньому збільшуються приблизно на 305.24.

RMSE моделі становить приблизно 10711.00, що є досить великою помилкою. На графіку видно значний розкид реальних значень навколо лінії регресії. Також помітно, що для однакового віку charges можуть дуже сильно відрізнятися.

Тому я не використовувала б цю модель у компанії як фінальну модель прогнозування. Вік сам по собі недостатньо добре пояснює медичні витрати курців.

Цю модель можна використати як базову для порівняння. Для покращення прогнозу варто додати інші ознаки, наприклад bmi, sex, children і region.